# Ready-to-React 모델 구조/shape 워크스루

이 노트북은 [Ready-to-React (ICLR 2025)](https://github.com/zju3dv/ready_to_react) 의
`ReactiveARDiffDecModel` 을 학습 데이터 없이 **구조와 shape 만** 공부하기 위한 파일입니다.

파이프라인은 세 단계로 나뉩니다.

1. **VQ-VAE 모션 토크나이저** ― 한 에이전트의 pose+root 시퀀스를 *temporal stride 4* 로
   압축해 codebook index 로 양자화합니다. (`configs/exps/vqvae_tokenizer.yaml`)
2. **AR 디퓨전 네트워크 (next latent predictor)** ― 상대방(oppo) 포즈와 과거 latent 를
   조건으로 받아, 다음 단위시간의 latent 를 denoise 합니다. (`ReactiveARDiffDecNetwork`)
3. **온라인 모션 디코더** ― 직전 U=4 프레임의 pose / root_info 와 직전·예측 latent 를 받아
   다음 U=4 프레임의 pose+root 를 연속 값으로 복원합니다. (`ReactiveARDecoder`)

> **전제**: DuoBox 데이터셋/사전학습 체크포인트는 없습니다. 모든 네트워크는 **random init**
> 으로 GPU 에 올려 shape 만 확인합니다.

### 자주 헷갈리는 dimension 정리

이 코드에는 **서로 다른 root 신호 3 종류** 가 등장합니다. 처음에 한 번만 정리해 두면
이후 cell 들이 훨씬 잘 읽힙니다.

| 이름            | shape (마지막 차원) | 무엇인가                                                     | 참조 |
|-----------------|--------------------|-------------------------------------------------------------|------|
| `pose_series`   | **256**            | 자기 자신의 (joint pose 252) + (자기 root_ctrl 4) 합본       | 논문 Eq. 2 |
| `oppo_pose`     | **252**            | 상대방의 joint pose/rot/vel — root 정보 없음                  | 논문 Eq. 3 |
| `root_info`     | **5**              | 디코더용 보조 root 신호 = root2oppo_off(2) + root2oppo_dir(2) + 링중심거리(1) | 논문 Eq. 5 |

```python
# 한 줄 요약
agent pose_series: 256 = 252 joint features + 4 root_ctrl
opponent pose:     252 = 21 joints × (pos3 + rot6d + vel3)
decoder root_info:   5 = root2oppo_off2 + root2oppo_dir2 + ring_dist1
```

`pose_series` 끝의 **root_ctrl 4** 는 "직전 프레임의 *내 root* 기준 상대 이동/방향" 이고,
`root_info` 의 root2oppo 4 는 "직전 프레임의 *상대 root* 기준 상대 이동/방향" 입니다.
좌표계가 다르므로 둘은 별개의 신호입니다.

### 그 외 노테이션

* `B` = batch size, `T` = 프레임(30fps), `U=4` (unit_length, 토큰 하나가 4 프레임)
* `T_lat = T / U` (latent 시퀀스 길이, reactive model 은 `block_size=15` 를 사용)
* `J = 21` (Motive skeleton joint 수), `dpose = 12` (per-joint feature: pos3 + rot6d + vel3)


## 1. 환경 준비

`easyvolcap.engine` 은 import 시점에 `sys.argv` 를 읽어 전역 `cfg` 를 만듭니다. 그래서 노트북에서도
원래의 CLI 와 동일하게 `-c configs/exps/reactive_model.yaml` 을 흉내 내야 합니다.

또한 리포지토리에는 norm 통계 `.npy` 파일이 없으므로 shape 만 일치하는 **더미 파일** 을 만들어
둡니다 (mean=0, std=1). 실제 학습/추론에는 데이터셋 전처리로 만들어진 파일을 써야 합니다.


In [1]:
import os, sys
import numpy as np
from pathlib import Path

REPO = Path('/workspace/ready_to_react')
os.chdir(REPO)                      # cfg 내 상대경로 기준을 맞춤
sys.path.insert(0, str(REPO))

# ---- easyvolcap 이 import 시점에 읽는 CLI 를 가짜로 세팅 ----
sys.argv = ['notebook', '-c', 'configs/exps/reactive_model.yaml', '-t', 'test']

# ---- 더미 norm 파일 생성 (mean=0, std=1) ----
def ensure_norm(path: str, feats: int):
    p = REPO / path
    if p.exists():
        return
    p.parent.mkdir(parents=True, exist_ok=True)
    stats = np.zeros((2, feats), dtype=np.float32)
    stats[1] = 1.0                  # std = 1
    np.save(p, stats)
    print(f'  created {p.relative_to(REPO)}  shape={stats.shape}')

ensure_norm('data/boxing/reactive/motoken/pose_series.npy', 256)
ensure_norm('data/boxing/reactive/react_motoken/oppo_pose.npy', 252)
ensure_norm('data/boxing/reactive/react_motoken/root_info.npy', 5)
print('norm files ready.')

norm files ready.


## 2. `cfg` 로드와 모듈 레지스트리 등록

`discover_modules()` 가 `reactmotion/{models,networks,dataloaders,...}` 을 import 해서
`MODELS`, `NETWORKS`, `ENCODERS`, `DECODERS` 레지스트리를 채웁니다.


In [2]:
from easyvolcap.engine import cfg
from reactmotion.utils.engine_utils import discover_modules
discover_modules()

print('exp_name          :', cfg.exp_name)
print('model_cfg.type    :', cfg.model_cfg.type)
print('network_cfg.type  :', cfg.model_cfg.network_cfg.type)
print('decoder_cfg.type  :', cfg.model_cfg.decoder_cfg.type)
print('motoken_cfg_file  :', cfg.motoken_cfg_file)
print('window_cfg        :', dict(cfg.window_cfg))

# motion_repr_transform 은 import 시 motoken_cfg 를 읽어 U, D 를 계산함
from reactmotion.utils.motion_repr_transform import U, D, njoints, dpose
print(f'\nU (unit_length)    = {U}  (1 토큰 = {U} 프레임)')
print(f'J (njoints)        = {njoints}')
print(f'dpose / joint       = {dpose}  (pos3 + rot6d + vel3)')
print(f'D = J * dpose      = {D}')
print(f'pose_series feats  = D + 4 (root ctrl) = {D+4}')

exp_name          : reactive_model
model_cfg.type    : ReactiveARDiffDecModel
network_cfg.type  : ReactiveARDiffDecNetwork
decoder_cfg.type  : ReactiveARDecoder
motoken_cfg_file  : configs/exps/vqvae_tokenizer.yaml
window_cfg        : {'vocab_size': 512, 'vqmotion_window_size': 64, 'unit_length': 4, 'block_size': 15, 'infer_gpttoken_window_size': 1}

U (unit_length)    = 4  (1 토큰 = 4 프레임)
J (njoints)        = 21
dpose / joint       = 12  (pos3 + rot6d + vel3)
D = J * dpose      = 252
pose_series feats  = D + 4 (root ctrl) = 256


## 3. VQ-VAE 모션 토크나이저

`VQVAEEMAResetT2MGPTMoTokenNetwork` 는 T2M-GPT 의 VQ-VAE 구조를 그대로 사용합니다.

```
pose_series  (B, T, 256)          # 한 에이전트의 로컬 pose + root_ctrl
   │  normalize (Xnorm)
   ▼
Encoder 1D-conv + ResNet, down_t=2, stride_t=2   (총 4배 다운샘플)
   │
   ▼
z_e_x        (B, T/U, 512)
   │
   ▼
QuantizeEMAReset  ─ codebook (K=512, dim=512)
   │
   ▼
code_idx     (B, T/U)   +   z_q_x (B, T/U, 512)
   │
   ▼
Decoder 1D-conv + ResNet, Upsample×2 두 번
   │
   ▼
x_tilde      (B, T, 256)
```

VQ-VAE 사전학습은 `vqmotion_window_size=64` 프레임 (= 16 토큰) 으로 잘라 진행하지만,
reactive 단계의 latent 시퀀스는 `block_size=15` 토큰 (= 60 프레임) 을 사용합니다.
이 노트북에서는 reactive 쪽과 동일하게 `T = 15 * U = 60` 으로 한 번 통과시킵니다.


In [3]:
import torch
from easyvolcap.engine import NETWORKS
from reactmotion.utils.engine_utils import parse_args_list
from easyvolcap.utils.base_utils import dotdict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# motoken 용 cfg 는 별도 yaml 을 파싱
motoken_cfg = parse_args_list(['-c', cfg.motoken_cfg_file])
print('motoken input_dim :', motoken_cfg.model_cfg.network_cfg.input_dim)
print('motoken dim (lat) :', motoken_cfg.model_cfg.network_cfg.dim)
print('codebook K        :', motoken_cfg.model_cfg.network_cfg.K)

motoken_net = NETWORKS.build(motoken_cfg.model_cfg.network_cfg).to(device).eval()
n_params = sum(p.numel() for p in motoken_net.parameters()) / 1e6
print(f'motoken params    : {n_params:.2f} M')

device: cuda


motoken input_dim : 256
motoken dim (lat) : 512
codebook K        : 512


motoken params    : 19.42 M


In [4]:
B, T_lat = 2, 15          # reactive model 의 block_size 와 동일
T_pose = T_lat * U         # = 60 프레임

# shape-only 확인용 랜덤 pose_series
pose_series = torch.randn(B, T_pose, 256, device=device)
print('input  pose_series :', tuple(pose_series.shape))

with torch.no_grad():
    # 내부 연산을 단계별로 따라가 봅니다
    x = motoken_net.preprocess(pose_series)
    print('  after Normalize  :', tuple(x.shape))
    z_e_x = motoken_net.encoder(x)
    print('  after Encoder    :', tuple(z_e_x.shape), '   # T 가 U배 다운샘플')

    # quantize & dequantize
    Bz, Tz, Cz = z_e_x.shape
    # 코드북은 학습 첫 iter 에서 init 되지만, 랜덤 가중치 상태에서는 0벡터라 nearest 가
    # 모두 동일해집니다. 여기서는 shape 확인 용도로 코드북을 랜덤으로 채워 사용합니다.
    motoken_net.quantizer.codebook = torch.randn(
        motoken_net.quantizer.nb_code, motoken_net.quantizer.code_dim, device=device)
    code_idx = motoken_net.quantizer.quantize(z_e_x.view(-1, Cz)).view(Bz, Tz)
    print('  code_idx         :', tuple(code_idx.shape), '   # 정수 토큰 인덱스')
    z_q_x = motoken_net.quantizer.dequantize(code_idx)
    print('  z_q_x (dequant)  :', tuple(z_q_x.shape))

    x_tilde = motoken_net.decoder(z_q_x)
    x_tilde = motoken_net.postprocess(x_tilde)
    print('output x_tilde     :', tuple(x_tilde.shape))

input  pose_series : (2, 60, 256)
  after Normalize  : (2, 60, 256)


  after Encoder    : (2, 15, 512)    # T 가 U배 다운샘플
  code_idx         : (2, 15)    # 정수 토큰 인덱스
  z_q_x (dequant)  : (2, 15, 512)
output x_tilde     : (2, 60, 256)


`encode()` / `decode()` 는 위 단계를 한 번에 감싸 둔 헬퍼입니다. 이후 reactive 단계에서는 이
`encode()` 로부터 얻은 **latent (B, T_lat, 512)** 또는 **code_idx (B, T_lat)** 가 입력이 됩니다.


In [5]:
with torch.no_grad():
    tokens = motoken_net.encode(pose_series)        # (B, T_lat)
    latents = motoken_net.quantizer.dequantize(tokens)   # (B, T_lat, 512)
    recon   = motoken_net.decode(tokens)           # (B, T_pose, 256)
print('tokens  :', tuple(tokens.shape), tokens.dtype)
print('latents :', tuple(latents.shape))
print('recon   :', tuple(recon.shape))

tokens  : (2, 15) torch.int64
latents : (2, 15, 512)
recon   : (2, 60, 256)


## 4. Reactive AR Diffusion Network

`ReactiveARDiffDecNetwork` 는 두 단계로 호출됩니다.

1. `conditon_forward(cond_oppo, cond_latents, mask)` ― 한 sequence 에 대해 **딱 한 번** 실행해
   `conditions` 를 얻습니다. (Transformer encoder, causal mask)
2. `forward(x_t, timesteps, conditions)` ― 디퓨전 스텝마다 호출되어 **노이즈 latent 를 x_start 로**
   denoise 합니다 (config 의 `predict_xstart=True`).

### shape 흐름

```
cond_oppo      (B, T_lat, 252)    # oppo 의 과거 pose, U-stride 로 다운샘플된 상태
cond_latents   (B, T_lat, 512)    # 자기 자신의 과거 latent (학습 시 target[:, :-1])
           │
           ▼  (각각 Linear -> 512, [T,B,C] 배치로 rearrange, 더하기)
xseq           (T_lat, B, 512)
           │  PositionalEnc + causal TransformerEncoder
           ▼
conditions     (B, T_lat, 512)
```

그리고 denoise 한 번은 다음 흐름을 따릅니다.

```
x_t           (B, T_lat, 512)        # 노이즈 섞인 latent
timesteps     (B, T_lat)              # diffusion step
   │        TimestepEmbedder -> (B, T_lat, 512)
   │        conditions + t_emb
cat(x_t, conditions)  (B, T_lat, 1024)
   │        mlps: Linear 1024 -> 512
   ▼        output_process: Linear 512 -> 512
output (x_start 예측)  (B, T_lat, 512)
```

> 참고: `conditon_forward` 의 인자 `mask` 는 padding mask 로 `key_padding_mask` 에 들어가고,
> causal 관계는 별도의 `subsequent_mask` 가 처리합니다.


In [6]:
diff_net = NETWORKS.build(cfg.model_cfg.network_cfg).to(device).eval()
print('ReactiveARDiffDecNetwork:')
print('  input_feats (latent)  :', diff_net.input_feats)
print('  oppo_feats             :', diff_net.oppo_feats)
print('  latent_dim             :', diff_net.latent_dim)
print('  num_layers / heads     :', diff_net.num_layers, '/', diff_net.num_heads)
print('  params                 : {:.2f} M'.format(
    sum(p.numel() for p in diff_net.parameters())/1e6))

ReactiveARDiffDecNetwork:
  input_feats (latent)  : 512
  oppo_feats             : 252
  latent_dim             : 512
  num_layers / heads     : 8 / 4
  params                 : 18.79 M


In [7]:
# 4-1. 조건 인코딩 한 번
cond_oppo    = torch.randn(B, T_lat, 252, device=device)
cond_latents = torch.randn(B, T_lat, 512, device=device)

with torch.no_grad():
    conditions = diff_net.conditon_forward(cond_oppo, cond_latents, mask=None)
print('conditions :', tuple(conditions.shape),  '   # (B, T_lat, latent_dim)')

# 4-2. 한 디퓨전 스텝
x_t       = torch.randn(B, T_lat, 512, device=device)
timesteps = torch.randint(0, 1000, (B, T_lat), device=device)    # long

with torch.no_grad():
    x_start_pred = diff_net(x_t, timesteps, conditions)
print('x_start_pred:', tuple(x_start_pred.shape))

conditions : (2, 15, 512)    # (B, T_lat, latent_dim)
x_start_pred: (2, 15, 512)


## 5. 온라인 모션 디코더 (`ReactiveARDecoder`)

디퓨전이 뽑은 `future_latent` 는 `U=4` 프레임을 **하나로 뭉친** 표현입니다. 이걸 다시
**연속값의 pose_series 와 root_info** 로 풀어내는 것이 디코더의 역할입니다.

논문 표현으로는 "past pose/root + 직전 latent + 현재 latent → 다음 U 프레임의 pose/root"
입니다. 즉 4 가지 조각을 입력으로 받아 다음 U 프레임을 회귀합니다.

```
past_pose_series  (B, U=4, 256)       Linear 256->512  -> (4, B, 512)
past_root_info    (B, U=4, 5)         Linear 5  ->512  -> (4, B, 512)
past_latents      (B, 1, 512)         Linear 512->512  -> (1, B, 512)
future_latents    (B, 1, 512)         Linear 512->512  -> (1, B, 512)
                                       │  토큰 축으로 concat
                                       ▼
x                 (4+4+1+1=10, B, 512)
PositionalEnc + (양방향) TransformerEncoder -> output (10, B, 512)

pose_head  (output[0:4])   -> (B, 4, 256)   # 다음 U 프레임 pose_series
root_head  (output[4:8])   -> (B, 4, 5)     # 다음 U 프레임 root_info
```

> **코드 관점의 보조 설명**: 출력은 "입력 pose/root 와 같은 위치(슬롯)" 에서 읽어 오므로,
> 코드 구조상 그 슬롯들이 **query 처럼** 동작한다고 볼 수 있습니다. 다만 이 표현은 논문에
> 등장하는 용어는 아닙니다. 논문은 단순히 "4 가지 입력으로 next U frames 를 decode" 라고
> 설명합니다.

학습 시에는 이 디코더를 한 번 호출해 14 개의 (block 단위) 예측을 동시에 만들고, 추론 시에는
가장 최신 1 개 block 만 예측합니다 — 자세한 shape 차이는 6-1 / 6-2 에서 비교합니다.


In [8]:
dec_net = NETWORKS.build(cfg.model_cfg.decoder_cfg).to(device).eval()
print('ReactiveARDecoder:')
print('  pose_feats    :', dec_net.pose_feats)
print('  root_feats    :', dec_net.root_feats)
print('  latents_feats :', dec_net.latents_feats)
print('  params        : {:.2f} M'.format(
    sum(p.numel() for p in dec_net.parameters())/1e6))

past_pose_series = torch.randn(B, U, 256, device=device)
past_root_info   = torch.randn(B, U, 5,   device=device)
past_latents     = torch.randn(B, 1, 512, device=device)
future_latents   = torch.randn(B, 1, 512, device=device)

with torch.no_grad():
    pred_pose, pred_root = dec_net(past_pose_series, past_root_info,
                                   past_latents, future_latents)
print('pred_pose_series :', tuple(pred_pose.shape), '  # 다음 U 프레임')
print('pred_root_info   :', tuple(pred_root.shape))

ReactiveARDecoder:
  pose_feats    : 256
  root_feats    : 5
  latents_feats : 512
  params        : 17.35 M
pred_pose_series : (2, 4, 256)   # 다음 U 프레임
pred_root_info   : (2, 4, 5)


## 6. 전체 모델 구축

위의 세 네트워크를 담은 `ReactiveARDiffDecModel` 을 실제로 빌드합니다. 이 모델은 다음을
묶어 둡니다.

* `diffusion_forward` ― **학습 시** x_start 를 q_sample 로 노이즈화 → 네트워크로 예측 →
  MSE loss.
* `diffusion_inference` ― **추론 시** DDIM loop 로 `sample_steps` (기본 50) 만큼 denoise.
* `decoder_forward` ― 디퓨전이 뽑은 미래 latent 를 연속값 pose 로 디코드.

또한 `Xnorm / Cnorm / Rnorm` (3. 에서 만든 norm 파일) 이 parameter buffer 로 등록됩니다.

> 학습과 추론은 **batch shape 가 다릅니다**. 6-1 / 6-2 에서 따로 살펴봅니다.


In [9]:
from easyvolcap.engine import MODELS

model = MODELS.build(cfg.model_cfg).to(device).eval()
print('model class     :', type(model).__name__)
print('diffusion steps :', model.diffusion_steps,
      '  skip_steps(=steps - sample_steps):', model.skip_steps)
print('predict_xstart  :', model.predict_xstart)
print('Xnorm shape     :', tuple(model.Xnorm.shape), ' (2, 256)')
print('Cnorm shape     :', tuple(model.Cnorm.shape), ' (2, 252)')
print('Rnorm shape     :', tuple(model.Rnorm.shape), ' (2, 5)')
total = sum(p.numel() for p in model.parameters()) / 1e6
print(f'model params    : {total:.2f} M')

2026-04-24 15:03:45.978728  load norm file          ]8;id=14160162;file:///workspace/ready_to_react/reactmotion/utils/motion_repr_transform.py\motion_repr_transform.py]8;;\:]8;id=14160163;file:///workspace/ready_to_react/reactmotion/utils/motion_repr_transform.py#194\194]8;;\
                           data/boxing/reactive/mot                             
                           oken/pose_series.npy                                 

2026-04-24 15:03:45.984893  load norm file          ]8;id=14160168;file:///workspace/ready_to_react/reactmotion/utils/motion_repr_transform.py\motion_repr_transform.py]8;;\:]8;id=14160169;file:///workspace/ready_to_react/reactmotion/utils/motion_repr_transform.py#194\194]8;;\
                           data/boxing/reactive/rea                             
                           ct_motoken/oppo_pose.npy                             

2026-04-24 15:03:45.988142  load norm file          ]8;id=14160174;file:///workspace/ready_to_react/reactmotion/utils/motion_repr_transform.py\motion_repr_transform.py]8;;\:]8;id=14160175;file:///workspace/ready_to_react/reactmotion/utils/motion_repr_transform.py#194\194]8;;\
                           data/boxing/reactive/rea                             
                           ct_motoken/root_info.npy                             

model class     : ReactiveARDiffDecModel
diffusion steps : 1000   skip_steps(=steps - sample_steps): 950
predict_xstart  : True
Xnorm shape     : (2, 256)  (2, 256)
Cnorm shape     : (2, 252)  (2, 252)
Rnorm shape     : (2, 5)  (2, 5)
model params    : 36.14 M


### 6-1. 학습(Training) 경로

학습 batch 는 dataloader (`BoxingReactDataset`) 가 만들어 줍니다. 한 windowing 안의
정보를 **모두 frame-level 로** 담아 두고, 모델 내부에서 필요할 때 latent step 으로
잘라 씁니다.

#### 학습 batch shape

| key            | shape               | 설명 |
|----------------|---------------------|------|
| `latents`      | (B, **15**, 512)    | VQ-VAE 가 인코드한 codebook latent (latent step) |
| `oppo_pose`    | (B, **60**, 252)    | 상대방 pose, **frame 단위** (정규화 전) |
| `pose_series`  | (B, **60**, 256)    | 자기 pose+root_ctrl, frame 단위 |
| `root_info`    | (B, **60**, 5)      | 디코더용 root aux 신호, frame 단위 |
| `mask`         | (B, 15)             | latent 시퀀스 padding mask |
| `pose_mask`    | (B, 60)             | pose 시퀀스 padding mask |

`block_size = 15`, `U = 4` 라서 latent 축은 15, frame 축은 60 입니다.

#### 모델 내부에서 일어나는 일

1. `target = batch.latents` 를 **시간으로 1칸 밀어** AR 학습:
   - `target[:, :-1]` (B, 14, 512) → 입력
   - `target[:, 1:]`  (B, 14, 512) → x_start (예측 타깃)
2. 디퓨전 condition 인코딩에서 `oppo_pose` 는 `[:, :-U:U]` 로 **U-stride 다운샘플** →
   (B, 60, 252) → (B, 14, 252) 로 latent 축에 맞춰집니다.
3. 디코더 입력은 `pose_series[:, :-U]` (B, 56, 256) 을 4 프레임씩 잘라 **(B*14, 4, 256)** 로
   reshape 합니다. latent 도 한 토큰씩 잘라 **(B*14, 1, 512)** 로 reshape 한 뒤 배치 차원에
   합쳐 한 번에 처리됩니다. 손실은 `pose_series[:, U:]` (B, 56, 256) 과 비교합니다.

> 즉 학습은 **"한 윈도우 안의 14 개 block 을 병렬"** 로 한 번에 학습합니다.


In [10]:
# 학습 경로를 shape 만 돌려보기 위해 BoxingReactDataset 출력과 동일한 shape 로 fake batch 생성
L  = T_lat                       # = block_size = 15
Tp = L * U                       # = 60

batch = dotdict(
    latents     = torch.randn(B, L,  512, device=device),
    oppo_pose   = torch.randn(B, Tp, 252, device=device),
    pose_series = torch.randn(B, Tp, 256, device=device),
    root_info   = torch.randn(B, Tp, 5,   device=device),
    mask        = torch.ones (B, L,        device=device),
    pose_mask   = torch.ones (B, Tp,       device=device),
)
print('batch shapes:')
for k, v in batch.items():
    print(f'  {k:<12s} {tuple(v.shape)}')

# normalize_batch_data 는 Xnorm/Cnorm/Rnorm 으로 인플레이스 정규화
model.train()                   # training-time branch 를 타도록
model.normalize_batch_data(batch)

target = batch.latents          # (B, 15, 512)

# (1) 디퓨전 쪽 forward
#   - x_start    = target[:, 1:]   -> (B, 14, 512)
#   - past_target = target[:, :-1] -> (B, 14, 512)
#   - oppo_pose 는 안에서 [:, :-U:U] 로 stride 됩니다 -> (B, 14, 252)
diff_out, diff_loss = model.diffusion_forward(target, batch)
print('\n--- diffusion_forward ---')
print('diffusion_output :', tuple(diff_out.shape), '  # x_start 예측 = target[:, 1:]')
print('diff_recon_loss  :', float(diff_loss.detach()))

# (2) 디코더 쪽 forward
#   - 14 개의 슬롯을 배치 차원에 합쳐 (B*14, ...) 로 한 번에 처리
#   - 손실은 batch.pose_series[:, U:] (B, 56, 256) 과 비교합니다
dec_pose_loss, dec_root_loss = model.decoder_forward(target[:, :-1], diff_out, batch)
print('\n--- decoder_forward ---')
print('dec_pose_rec_loss:', tuple(dec_pose_loss.shape), 'mean=', float(dec_pose_loss.mean()))
print('dec_root_rec_loss:', tuple(dec_root_loss.shape), 'mean=', float(dec_root_loss.mean()))

batch shapes:
  latents      (2, 15, 512)
  oppo_pose    (2, 60, 252)
  pose_series  (2, 60, 256)
  root_info    (2, 60, 5)
  mask         (2, 15)
  pose_mask    (2, 60)

--- diffusion_forward ---
diffusion_output : (2, 14, 512)   # x_start 예측 = target[:, 1:]
diff_recon_loss  : 1.1327780485153198

--- decoder_forward ---
dec_pose_rec_loss: (2, 56) mean= 1.3036565780639648
dec_root_rec_loss: (2, 56) mean= 1.2650576829910278


### 6-2. 추론(Inference) 경로

추론은 **매 step 마다 next 1 block(=U 프레임) 만** 예측하고, 그 결과를 다시 agent 의
버퍼에 누적해 다음 step 에서 사용합니다 (streaming generation).

학습 batch 와 다른 점이 두 가지 있습니다.

* `oppo_pose` 가 **이미 U-stride 로 다운샘플된 상태** 로 들어옵니다. → frame 축이 아니라
  latent 축 (T_lat) 입니다.
* `pose_series` / `root_info` 는 윈도우 전체가 아니라 **직전 U=4 프레임** 만 들어옵니다.

#### 추론 batch shape

| key            | shape               | 설명 |
|----------------|---------------------|------|
| `latents`      | (B, **T_lat**, 512) | 지난 T_lat 개 latent (T_lat ≤ 15) |
| `oppo_pose`    | (B, **T_lat**, 252) | 지난 T_lat block 의 상대 pose (이미 U-stride) |
| `pose_series`  | (B, **U=4**, 256)   | 직전 U 프레임의 자기 pose |
| `root_info`    | (B, **U=4**, 5)     | 직전 U 프레임의 root_info |

#### 처리 흐름

1. `condition_forward` 가 `oppo_pose / latents` 를 받아 conditions (B, T_lat, 512) 생성.
2. `ddim_sample_loop` 가 50 step 동안 denoise → future_latents (B, T_lat, 512).
3. 그 중 **마지막 1 개** `future_latents[:, -1:]` 만 디코더로 보내 **다음 U=4 프레임** 의
   pose_series / root_info 를 복원합니다.

DDIM 전체 50 step 은 노트북에서 시간이 걸리므로, 데모에서는 `sample_steps=5` 로 줄여
빠르게 확인합니다.

#### 학습 vs 추론 한 눈에 보기

```
                     학습              추론
oppo_pose            (B, 60, 252)      (B, T_lat, 252)   ← 이미 stride
pose_series          (B, 60, 256)      (B, 4,     256)   ← 직전 U 프레임
root_info            (B, 60, 5)        (B, 4,     5)
latents              (B, 15, 512)      (B, T_lat, 512)
```


In [11]:
model.eval()
# sample_steps 를 5 로 줄여 반복 계산량을 낮춥니다
model.skip_steps = model.diffusion_steps - 5
print('sample_steps (effective) =', model.diffusion_steps - model.skip_steps)

# 추론 시 batch 는 *현재 윈도우* 만 들어 있습니다. `ReactiveARDiffDecModel.network_inference`
# 가 기대하는 형태:
#   latents    : (B, T_lat, 512)           past latents
#   oppo_pose  : (B, T_lat, 252)           past oppo pose (이미 U-stride 로 샘플된 상태)
#   pose_series: (B, U,    256)            직전 U 프레임
#   root_info  : (B, U,    5)              직전 U 프레임
infer_batch = dotdict(
    latents     = torch.randn(B, T_lat, 512, device=device),
    oppo_pose   = torch.randn(B, T_lat, 252, device=device),
    pose_series = torch.randn(B, U,     256, device=device),
    root_info   = torch.randn(B, U,     5,   device=device),
)
print('infer_batch shapes:')
for k, v in infer_batch.items():
    print(f'  {k:<12s} {tuple(v.shape)}')

with torch.no_grad():
    pred_lat, pred_pose, pred_root = model.network_inference(infer_batch)
print('\n--- network_inference (1 step) ---')
print('pred_latents     :', tuple(pred_lat.shape),  '  # 다음 U 프레임 분의 latent 1개')
print('pred_pose_series :', tuple(pred_pose.shape), '  # 다음 U 프레임 pose_series (denorm)')
print('pred_root_info   :', tuple(pred_root.shape), '  # 다음 U 프레임 root_info (denorm)')

sample_steps (effective) = 5
infer_batch shapes:
  latents      (2, 15, 512)
  oppo_pose    (2, 15, 252)
  pose_series  (2, 4, 256)
  root_info    (2, 4, 5)

--- network_inference (1 step) ---
pred_latents     : (2, 1, 512)   # 다음 U 프레임 분의 latent 1개
pred_pose_series : (2, 4, 256)   # 다음 U 프레임 pose_series (denorm)
pred_root_info   : (2, 4, 5)   # 다음 U 프레임 root_info (denorm)


## 7. pose_series 의 252/256 채널을 분해하기

`MoReprTrans.split_pose` 는 256-d pose_series 의 각 채널이 무엇인지 보여줍니다.

```
[ 0 : D=252 ]  per-joint 12 = pos(3) + rot6d(6) + vel(3)     ─ 21개 관절 × 12
[ D : D+2  ]  root_off   (x, z)    ─ 직전 프레임 root 좌표계에서의 상대 이동
[D+2: D+4  ]  root_dir   (dx, dz)  ─ 직전 프레임 root 좌표계에서의 상대 방향
```

이 표현 방식이 정확히 "**로컬 좌표계 per-frame**" 으로 고정되어 있어 AR 롤아웃이 가능합니다.


In [12]:
from reactmotion.utils.motion_repr_transform import MoReprTrans

loc = MoReprTrans.split_pose(pred_pose)           # pred_pose: (B, U, 256)
for k, v in loc.items():
    print(f'{k:<10s} : {tuple(v.shape)}')

pose_pos   : (2, 4, 21, 3)
pose_rot   : (2, 4, 21, 3, 3)
pose_vel   : (2, 4, 21, 3)
root_off   : (2, 4, 2)
root_dir   : (2, 4, 2)
root_ctrl  : (2, 4, 4)


## 8. 한 페이지 요약

* **온라인 reaction 정책** 은 "다음 U=4 프레임 분 latent 를 디퓨전으로 뽑고, 그 latent 를
  온라인 디코더가 실시간 pose 로 풀어내는" 구조입니다.
* VQ-VAE 는 **pose tokenizer** 역할만 합니다. codebook 은 학습 후 고정됩니다.
* 디퓨전 네트워크의 시퀀스 축은 `T_lat=15` 로 짧게 유지됩니다 (학습 시 실제 처리 길이는 14).
  이는 조건 인코딩과 denoising 의 계산량을 줄여 online inference 에 유리합니다.
  > 논문이 강조하는 핵심은 *T_lat 의 길이* 가 아니라 "diffusion head + auto-regressive
  > model + online motion decoder 로 streaming generation 을 가능하게 한 것" 입니다.
* 디코더는 transformer encoder 한 개 + linear head 두 개로, 직전 U=4 프레임의 pose/root
  슬롯과 두 latent 를 함께 입력받아 다음 U 프레임 pose / root_info 를 복원합니다.
* **Sparse control 버전** (`SparseControlARDiffDecModel`) 은 VR3 조인트(머리 + 양손) 의
  pos3 + rot6d = `27` 차원 `ctrl_info` 를 추가 입력으로 사용합니다. 이 신호는 **next latent
  predictor 쪽 transformer 와 online decoder 쪽 transformer 둘 모두에 condition 으로
  들어갑니다.**

전체 파라미터(네트워크 합계) 는 random init 기준 대략 다음과 같습니다.


In [13]:
def params(m): return sum(p.numel() for p in m.parameters()) / 1e6
print(f'VQ-VAE motion tokenizer : {params(motoken_net):6.2f} M')
print(f'Reactive diffusion net   : {params(diff_net):6.2f} M')
print(f'Reactive AR decoder      : {params(dec_net):6.2f} M')
print(f'Full ReactiveARDiffDecModel (buffers 포함): {params(model):6.2f} M')

VQ-VAE motion tokenizer :  19.42 M
Reactive diffusion net   :  18.79 M
Reactive AR decoder      :  17.35 M
Full ReactiveARDiffDecModel (buffers 포함):  36.14 M
